# **(1) Ingession pipeline**

In [20]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma

# from sentence_transformers import SentenceTransformer

### **Configurations**

In [51]:
doc_path = "docs"
db_path = "db/chroma_db"
embedding_model_path = "../../../Data/HuggingFaceEmbeddings Models/all-mpnet-base-v2"

chunk_size = 800
chunk_overlap = 0

In [53]:
# import os 
# os.listdir(embedding_model_path)

### **Load documents**

In [28]:
print(f"Loading docs from '{doc_path}'")

loader = DirectoryLoader(
    path=doc_path,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'}
)

documents = loader.load()

if len(documents)==0:
    raise FileNotFoundError(f"No any .txt files in path '{doc_path}' directory")

Loading docs from 'docs'


In [29]:
for i, doc in enumerate(documents[:2]):
    print("\n###", type(doc))
    print(f"Doc {i+1}")
    print(f"Source : {doc.metadata['source']}")
    print(f"Content length : {len(doc.page_content)} characters\n")


### <class 'langchain_core.documents.base.Document'>
Doc 1
Source : docs\369_1773048532050.txt
Content length : 3998401 characters


### <class 'langchain_core.documents.base.Document'>
Doc 2
Source : docs\about us.txt
Content length : 4814 characters



### **Split documents into chunks**

In [55]:
text_splitter = CharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

chunks = text_splitter.split_documents(documents=documents) # List of Document()

Created a chunk of size 1469, which is longer than the specified 800
Created a chunk of size 1183, which is longer than the specified 800
Created a chunk of size 885, which is longer than the specified 800
Created a chunk of size 2295, which is longer than the specified 800
Created a chunk of size 1567, which is longer than the specified 800
Created a chunk of size 3516, which is longer than the specified 800
Created a chunk of size 917, which is longer than the specified 800
Created a chunk of size 2719, which is longer than the specified 800
Created a chunk of size 8402, which is longer than the specified 800
Created a chunk of size 1215, which is longer than the specified 800
Created a chunk of size 1273, which is longer than the specified 800
Created a chunk of size 5832, which is longer than the specified 800
Created a chunk of size 3623, which is longer than the specified 800
Created a chunk of size 882, which is longer than the specified 800
Created a chunk of size 7570, which i

In [40]:
for i, chunk in enumerate(chunks[0:5], 1):
    print(f"DOC : {i}")
    print(f"metadatac : {chunk.metadata}")
    print(f"Chunk length : {len(chunk.page_content)} chars")
    print(f"Chunk content : {chunk.page_content}")
    print("-" * 50)

    if len(chunks) > 5:
        print(f"more {len(chunks) -5} are there more...")

DOC : 1
metadatac : {'source': 'docs\\369_1773048532050.txt'}
Chunk length : 739 chars
Chunk content : Stability beneath... acceleration ahead...


Commercial Bank of Ceylon PLC
Annual Report 2025
(Integrated Report and Financial Statements)
                                                 Best Bank in Sri Lanka 2025
                                                 Global Finance Magazine


                                                           Stability beneath... acceleration ahead...
                 Annual Report 2025
                 Commercial Bank of Ceylon PLC


www.combank.lk


                                                  Commercial Bank of Ceylon PLC
                                                  Annual Report 2025
                                                  (Integrated Report and Financial Statements)
--------------------------------------------------
more 2860 are there more...
DOC : 2
metadatac : {'source': 'docs\\369_1773048532050.txt'}
Chunk length : 1

### **Loading embedding model**

In [56]:
embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model_path,
    model_kwargs={'device':'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print("### Embedding model loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Embedding model loaded


### **Creating Vector DB**

In [60]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=db_path,
    collection_metadata={"hnsw:space" : "cosine"}
)

print("### vector db created")

### vector db created
